In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import rectools
from rectools import Columns

In [2]:
# Считываем датасеты

df_users = pd.read_csv('data_original/data_original/users.csv')
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_items = pd.read_csv('data_original/data_original/items.csv')

In [3]:
df_users = df_users.drop(df_users.index[-1])
df_interactions = df_interactions.drop(df_interactions.index[-1])
df_items = df_items.drop(df_items.index[-1])

In [4]:
df_items.head(5)

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [5]:
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')  # Convert to datetime, coerce invalid to NaT

In [6]:
# Добавление нового столбца
df_interactions['completed'] = (df_interactions['watched_pct'] >= 80).astype(int)

In [7]:
k=10

test_size_days=10

In [8]:
from datetime import datetime, timedelta

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

In [9]:
# Разделение на тренировочный и тестовый
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

In [10]:
# не обрабатываем холодных пользователей
df_interactions_test = df_interactions_test.loc[df_interactions_test["user_id"].isin(df_interactions_train["user_id"]) & df_interactions_test["item_id"].isin(df_interactions_train["item_id"])]

In [11]:
df_users[['user_id',"age","income"]]

,user_id,age,income
0,973171,age_25_34,income_60_90
1,962099,age_18_24,income_20_40
2,1047345,age_45_54,income_40_60
3,721985,age_45_54,income_20_40
4,704055,age_35_44,income_60_90
...,...,...,...
840191,365945,age_25_34,income_20_40
840192,339025,age_65_inf,income_0_20
840193,983617,age_18_24,income_20_40
840194,251008,NaN,NaN


In [12]:
from sklearn.preprocessing import LabelEncoder

# Кодируем категориальные признаки (но можно их закодировать прямо при построении датасета)
le_age = LabelEncoder()
le_inc = LabelEncoder()
le_sex = LabelEncoder()
df_users["age_en"] = le_age.fit_transform(df_users["age"])
df_users["income_en"] = le_inc.fit_transform(df_users["income"])
df_users["sex_en"] = le_inc.fit_transform(df_users["sex"])
df_users.head()

,user_id,age,income,sex,kids_flg,age_en,income_en,sex_en
0,973171,age_25_34,income_60_90,М,1,1,4,1
1,962099,age_18_24,income_20_40,М,0,0,2,1
2,1047345,age_45_54,income_40_60,Ж,0,3,3,0
3,721985,age_45_54,income_20_40,Ж,0,3,2,0
4,704055,age_35_44,income_60_90,Ж,0,2,4,0


In [13]:
df_interactions_test.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)


df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)

C:\Users\kanze\AppData\Local\Temp\ipykernel_2972\1242639805.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item,


In [14]:
from rectools import  dataset 

In [15]:
dataset_tr = dataset.Dataset.construct(df_interactions_train)

In [16]:
from rectools.models import LightFMWrapperModel
from lightfm import LightFM

model = LightFMWrapperModel(
        epochs=100,
        num_threads=10,
        model=LightFM(no_components = 50, learning_rate=0.001, loss='warp')
        )

c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [17]:
users_unique = df_interactions_test[Columns.User].unique()
model.fit(dataset_tr)
recos = model.recommend(
    users=users_unique,
    dataset=dataset_tr,
    k = 10,
    filter_viewed= True 
)

ValueError: Not all input values are finite. Check the input for NaNs and infinite values.

In [ ]:
recos